# Basins of attraction: Newton's method on \(z^n - 1\)

This notebook explores one of the simplest dynamical systems you can write down:
repeatedly apply Newton's method to a polynomial and ask which root each starting
point ends up at.

* **Attractor:** a root of the polynomial.
* **Basin:** the set of starting points that converge to that root.
* **Settling time:** how many iterations it takes to get close.
* **The boundary** between basins can be a simple line or a fractal tangle,
  depending on \(n\).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from math_vis.basins import (
    basin_map, complex_grid, render_basins, render_fli,
    uncertainty_exponent, basin_entropy, normalize_iters,
)
from math_vis import finish

# Tests run this notebook as plain Python; keep resolutions modest there.
HIGH_RES = os.environ.get("PYTEST_CURRENT_TEST") is None
SIZE = 800 if HIGH_RES else 300

## \(n = 2\): the smooth control

For \(z^2 - 1\) the two basins are separated by the imaginary axis: a straight
line. The boundary has box-counting dimension \(D = 1\), so the uncertainty
exponent \(\alpha = 2 - D = 1\).

In [ ]:
z0 = complex_grid((-2, 2, -2, 2), (SIZE, SIZE))
basin, iters, fli = basin_map(z0, n=2, max_iter=60, tol=1e-6)
img_z2 = render_basins(basin, iters, norm="log1p", gamma=0.7)
finish(img_z2, grain=2.0)

## \(n = 3\): three roots, tangled boundary

Add one more root and the boundary becomes a fractal. Zooming in does not make
it go away.

In [ ]:
basin3, iters3, fli3 = basin_map(z0, n=3, max_iter=80, tol=1e-6)
img_z3 = render_basins(basin3, iters3, norm="log1p", gamma=0.7)
finish(img_z3, grain=2.0)

## Fast Lyapunov indicator edge map

The FLI highlights places where the Newton map stretches nearby points apart —
exactly the basin boundary. It is a useful edge detector, but it has its own
limits: very slow regions and points that land on periodic cycles can also light
up.

In [ ]:
img_fli = render_fli(fli3, cmap="magma")
finish(img_fli, grain=2.0)

## Metrics: putting a number on "tangled"

* **Uncertainty exponent \(\alpha\):** for a box size \(\varepsilon\), measure the
  fraction \(P_{\text{both}}(\varepsilon)\) of boxes that contain more than one basin.
  For a straight boundary \(P \sim \varepsilon^1\); for a fractal boundary the
  exponent is smaller because the boundary fills more boxes.
* **Basin entropy:** Shannon entropy of basin labels inside boxes, scanned over
  box sizes.
* **Fast Lyapunov indicator:** logarithmic growth of the Newton-map derivative.

All three are resolution-dependent, so the notebook will show both their
successes and their limits.

In [ ]:
for n in [2, 3, 4]:
    b, _, _ = basin_map(z0, n=n, max_iter=80, tol=1e-6)
    res = uncertainty_exponent(b, box_sizes=[4, 8, 16, 32, 64])
    print(f"n={n}: alpha = {res['alpha']:.3f}")

*This is a draft. More sections will follow: higher \(n\) contact sheets,
settling-time normalisation, a zoom sequence, the float32 precision lesson, and
a wall-art render.*